In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    BartTokenizer, 
    BartForConditionalGeneration
)
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
# --- Dataset ---
class Text2TextDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=128):
        df = pd.read_csv(csv_path)
        df = df[df["parsed_effect"] != "Make this sequence reduce the risk of ovarian cancer."]
        self.texts = df["effect"].tolist()
        self.targets = df["parsed_effect"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Tokenize source and target
        inputs = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        targets = self.tokenizer(
            self.targets[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs.input_ids.squeeze(),
            "attention_mask": inputs.attention_mask.squeeze(),
            "labels": targets.input_ids.squeeze(),
        }

# --- Tokenizer & Model ---
model_name = "facebook/bart-large" # or your checkpoint
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

# --- Dataset ---
train_dataset = Text2TextDataset("/kaggle/input/teach-modelparse/output (2).csv", tokenizer)

# --- Data collator handles padding dynamically ---
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# --- Training arguments ---
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    save_total_limit=2,
    fp16=True,               # mixed precision on T4s
    logging_steps=50,
    save_steps=500,
    report_to="none",
    dataloader_num_workers=2,
    predict_with_generate=True,
    push_to_hub=False,
    load_best_model_at_end=False,
)

# --- Trainer (handles multi-GPU automatically) ---
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# --- Train! ---
trainer.train()

# --- Save model ---
trainer.save_model("./bart_finetuned_dna")
tokenizer.save_pretrained("./bart_finetuned_dna")

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BartTokenizer, BartForConditionalGeneration

# --- 1️⃣ Define Dataset ---
class EffectDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {"text": self.texts[idx]}

# --- 2️⃣ Collate function ---
def collate_fn(batch, tokenizer, max_input_length=128):
    texts = [item["text"] for item in batch]
    encodings = tokenizer(
        texts,
        padding="longest",
        truncation=True,
        max_length=max_input_length,
        return_tensors="pt"
    )
    return encodings

# --- 3️⃣ Batched inference with progress saving ---
def generate_batched_save(model, tokenizer, dataset, batch_size=16, max_new_tokens=50,
                          device='cuda', save_every=5, save_path='/kaggle/working/progress.csv'):
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                            collate_fn=lambda batch: collate_fn(batch, tokenizer))
    
    all_outputs = []
    total_batches = len(dataloader)
    model.to(device)
    model.eval()
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            # Generate sequences
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                top_k=50,
                top_p=0.9,
                temperature=0.8,
                num_beams=4,
                early_stopping=True
            )
            
            # Decode
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            all_outputs.extend(decoded)
            
            print(f"✅ Batch {batch_idx+1}/{total_batches} done")
            
            # Save intermediate progress
            if (batch_idx + 1) % save_every == 0:
                temp_df = pd.DataFrame({
                    'effect': dataset.texts[:len(all_outputs)],
                    'parsed_effects': all_outputs
                })
                temp_df.to_csv(save_path, index=False)
                print(f"💾 Progress saved to {save_path}")
    
    # Final save
    final_df = pd.DataFrame({
        'effect': dataset.texts,
        'parsed_effects': all_outputs
    })
    final_df.to_csv('/kaggle/working/output.csv', index=False)
    print("🎉 Finished all batches — saved to /kaggle/working/output.csv")
    
    return all_outputs

# --- 4️⃣ Usage ---
df = pd.read_csv('/kaggle/input/all-uniprot-isodata/merged.csv')
texts = df['effect'].tolist()
dataset = EffectDataset(texts)

tokenizer = BartTokenizer.from_pretrained("./bart_finetuned_dna")
model = BartForConditionalGeneration.from_pretrained("./bart_finetuned_dna")

generated_texts = generate_batched_save(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
    batch_size=16,             # adjust to fit your GPU memory
    max_new_tokens=50,
    device='cuda',
    save_every=50,              # saves every 5 batches
    save_path='/kaggle/working/progress.csv'
)
